2024 dataset

In [ ]:
# ===========================================================
# AUTO ML BASELINE — Joint Label Prediction (NO LEAKAGE)
# ===========================================================

!pip install autogluon imbalanced-learn

import os, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, balanced_accuracy_score,
    classification_report, confusion_matrix
)
from imblearn.over_sampling import SMOTE
from google.colab import files
from autogluon.tabular import TabularPredictor

RANDOM_STATE = 42

# -----------------------------
# Load dataset
# -----------------------------
path = "devx_2024_modellingDataset_FLOtop5_union.csv"
df = pd.read_csv(path)

TARGETS = ["Frustration", "JobSat", "JobPerspectiveClass"]
FEATURES = [c for c in df.columns if c not in TARGETS]

# basic cleaning
df = df.fillna(df.median(numeric_only=True))
for t in TARGETS:
    df[t] = df[t].astype(int)

# -----------------------------
# Create JOINT label
# -----------------------------
tuples = list(zip(df["Frustration"], df["JobSat"], df["JobPerspectiveClass"]))
joint_labels, uniques = pd.factorize(tuples)
df["joint"] = joint_labels

print("Number of joint classes:", len(uniques))

# -----------------------------
# Split FIRST (to avoid leakage)
# -----------------------------
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df["joint"]
)

print("Train size (original):", train_df.shape)
print("Test  size (original):", test_df.shape)

# -----------------------------
# SMOTE ONLY ON TRAIN
# -----------------------------
sm = SMOTE(random_state=RANDOM_STATE, k_neighbors=3)

X_train = train_df[FEATURES]
y_train_joint = train_df["joint"]

X_res, y_res = sm.fit_resample(X_train, y_train_joint)

# Decode joint labels back into (F, J, P) just for inspection / metrics
decoded_train = np.array([uniques[i] for i in y_res])

train_bal = pd.DataFrame(X_res, columns=FEATURES)
train_bal["joint"] = y_res
train_bal["Frustration"] = decoded_train[:, 0].astype(int)
train_bal["JobSat"] = decoded_train[:, 1].astype(int)
train_bal["JobPerspectiveClass"] = decoded_train[:, 2].astype(int)

print("Train size after SMOTE:", train_bal.shape)

# -----------------------------
# Prepare data for AutoML (NO TARGETS AS FEATURES)
# -----------------------------
# AutoGluon sees: FEATURES + joint(label)
train_ag = train_bal.drop(columns=TARGETS)

# For test, we KEEP targets for metrics & CSV, but DROP them for AutoGluon’s input
test_ag = test_df.drop(columns=TARGETS)

# -----------------------------
# AutoML training
# -----------------------------
predictor = TabularPredictor(
    label="joint",
    problem_type="multiclass",
    eval_metric="balanced_accuracy"
).fit(
    train_ag,
    presets="best_quality",   # you can change to "medium_quality_faster_train" if needed
    time_limit=3600,
    verbosity=2
)

# -----------------------------
# Predictions on TEST (no leakage)
# -----------------------------
test_df = test_df.copy()  # to be explicit

# Predict using the leakage-free test_ag (no target columns)
test_df["joint_pred"] = predictor.predict(test_ag)

# Decode joint predictions -> per-target predictions
joint_pred_int = test_df["joint_pred"].astype(int).to_numpy()
decoded_pred = np.array([uniques[i] for i in joint_pred_int])

test_df["Frustration_pred"] = decoded_pred[:, 0].astype(int)
test_df["JobSat_pred"] = decoded_pred[:, 1].astype(int)
test_df["JobPerspectiveClass_pred"] = decoded_pred[:, 2].astype(int)

# -----------------------------
# JOINT METRICS
# -----------------------------
y_true = test_df["joint"].astype(int)
y_pred = test_df["joint_pred"].astype(int)

acc = accuracy_score(y_true, y_pred)
f1  = f1_score(y_true, y_pred, average="macro")
prec = precision_score(y_true, y_pred, average="macro")
rec  = recall_score(y_true, y_pred, average="macro")
bal = balanced_accuracy_score(y_true, y_pred)

print("\n================= JOINT METRICS =================")
print(f"Accuracy:          {acc:.4f}")
print(f"Macro-F1:          {f1:.4f}")
print(f"Precision (macro): {prec:.4f}")
print(f"Recall (macro):    {rec:.4f}")
print(f"Balanced Accuracy: {bal:.4f}")

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm)
cm_df.to_csv("automl_confusion_matrix.csv", index=False)

# -----------------------------
# PER TARGET METRICS
# -----------------------------
metrics_log = []

def per_target(name, true_col, pred_col):
    yt = test_df[true_col].astype(int)
    yp = test_df[pred_col].astype(int)

    acc = accuracy_score(yt, yp)
    f1  = f1_score(yt, yp, average="macro")
    prec = precision_score(yt, yp, average="macro")
    rec  = recall_score(yt, yp, average="macro")
    bal = balanced_accuracy_score(yt, yp)

    print(f"\n----- {name} -----")
    print(f"Accuracy:      {acc:.4f}")
    print(f"Macro-F1:      {f1:.4f}")
    print(f"Precision:     {prec:.4f}")
    print(f"Recall:        {rec:.4f}")
    print(f"Balanced Acc:  {bal:.4f}")
    report = classification_report(yt, yp, digits=4)
    print(report)

    metrics_log.append(f"\n==== {name} ====\n")
    metrics_log.append(report)

per_target("Frustration", "Frustration", "Frustration_pred")
per_target("JobSat", "JobSat", "JobSat_pred")
per_target("JobPerspectiveClass", "JobPerspectiveClass", "JobPerspectiveClass_pred")

# -----------------------------
# SAVE EVERYTHING
# -----------------------------
# Row-level predictions + features + true targets
test_df.to_csv("automl_joint_predictions.csv", index=False)

# Metrics log as text file
with open("automl_metrics.txt", "w") as f:
    f.write("=== JOINT METRICS ===\n")
    f.write(f"Accuracy: {acc:.4f}\n")
    f.write(f"Macro-F1: {f1:.4f}\n")
    f.write(f"Precision: {prec:.4f}\n")
    f.write(f"Recall: {rec:.4f}\n")
    f.write(f"Balanced Accuracy: {bal:.4f}\n\n")

    for m in metrics_log:
        f.write(m)
        f.write("\n")

print("\n💾 Files generated:")
print("automl_joint_predictions.csv")
print("automl_metrics.txt")
print("automl_confusion_matrix.csv")

# Auto-download (Colab)
files.download("automl_joint_predictions.csv")
files.download("automl_metrics.txt")
files.download("automl_confusion_matrix.csv")


/tmp/ipython-input-4185377470.py:37: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  joint_labels, uniques = pd.factorize(tuples)


Number of joint classes: 27
Train size (original): (52349, 18)
Test  size (original): (13088, 18)


No path specified. Models will be saved in: "AutogluonModels/ag-20251202_203212"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Oct  2 10:42:05 UTC 2025
CPU Count:          2
Memory Avail:       9.58 GB / 12.67 GB (75.6%)
Disk Space Avail:   62.21 GB / 107.72 GB (57.8%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `num_stack_le

Train size after SMOTE: (637848, 18)


Leaderboard on holdout data (DyStack):
                    model  score_holdout  score_val        eval_metric  pred_time_test  pred_time_val    fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0  NeuralNetFastAI_BAG_L2       0.756882   0.711355  balanced_accuracy       30.508391      38.217114  751.992131                 3.758726                5.755108         164.316305            2       True          4
1     WeightedEnsemble_L3       0.756600   0.711507  balanced_accuracy       31.896813      40.577729  903.805340                 0.019497                0.346663          38.808245            3       True          6
2  NeuralNetFastAI_BAG_L1       0.727672   0.688009  balanced_accuracy        4.521947       5.574723  399.844968                 4.521947                5.574723         399.844968            1       True          1
3     WeightedEnsemble_L2       0.725683   0.690033  balanced_accuracy       26.779705      3

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

2025

In [ ]:
# ===========================================================
# AUTO ML BASELINE — Joint Label Prediction (NO LEAKAGE)
# ===========================================================

!pip install autogluon imbalanced-learn

import os, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, balanced_accuracy_score,
    classification_report, confusion_matrix
)
from imblearn.over_sampling import SMOTE
from google.colab import files
from autogluon.tabular import TabularPredictor

RANDOM_STATE = 42

# -----------------------------
# Load dataset
# -----------------------------
path = "devx_2025_modellingDataset_FLOtop5_union.csv"
df = pd.read_csv(path)

TARGETS = ["Frustration", "JobSat", "JobPerspectiveClass"]
FEATURES = [c for c in df.columns if c not in TARGETS]

# basic cleaning
df = df.fillna(df.median(numeric_only=True))
for t in TARGETS:
    df[t] = df[t].astype(int)

# -----------------------------
# Create JOINT label
# -----------------------------
tuples = list(zip(df["Frustration"], df["JobSat"], df["JobPerspectiveClass"]))
joint_labels, uniques = pd.factorize(tuples)
df["joint"] = joint_labels

print("Number of joint classes:", len(uniques))

# -----------------------------
# Split FIRST (to avoid leakage)
# -----------------------------
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df["joint"]
)

print("Train size (original):", train_df.shape)
print("Test  size (original):", test_df.shape)

# -----------------------------
# SMOTE ONLY ON TRAIN
# -----------------------------
sm = SMOTE(random_state=RANDOM_STATE, k_neighbors=3)

X_train = train_df[FEATURES]
y_train_joint = train_df["joint"]

X_res, y_res = sm.fit_resample(X_train, y_train_joint)

# Decode joint labels back into (F, J, P) just for inspection / metrics
decoded_train = np.array([uniques[i] for i in y_res])

train_bal = pd.DataFrame(X_res, columns=FEATURES)
train_bal["joint"] = y_res
train_bal["Frustration"] = decoded_train[:, 0].astype(int)
train_bal["JobSat"] = decoded_train[:, 1].astype(int)
train_bal["JobPerspectiveClass"] = decoded_train[:, 2].astype(int)

print("Train size after SMOTE:", train_bal.shape)

# -----------------------------
# Prepare data for AutoML (NO TARGETS AS FEATURES)
# -----------------------------
# AutoGluon sees: FEATURES + joint(label)
train_ag = train_bal.drop(columns=TARGETS)

# For test, we KEEP targets for metrics & CSV, but DROP them for AutoGluon’s input
test_ag = test_df.drop(columns=TARGETS)

# -----------------------------
# AutoML training
# -----------------------------
predictor = TabularPredictor(
    label="joint",
    problem_type="multiclass",
    eval_metric="balanced_accuracy"
).fit(
    train_ag,
    presets="best_quality",   # you can change to "medium_quality_faster_train" if needed
    time_limit=3600,
    verbosity=2
)

# -----------------------------
# Predictions on TEST (no leakage)
# -----------------------------
test_df = test_df.copy()  # to be explicit

# Predict using the leakage-free test_ag (no target columns)
test_df["joint_pred"] = predictor.predict(test_ag)

# Decode joint predictions -> per-target predictions
joint_pred_int = test_df["joint_pred"].astype(int).to_numpy()
decoded_pred = np.array([uniques[i] for i in joint_pred_int])

test_df["Frustration_pred"] = decoded_pred[:, 0].astype(int)
test_df["JobSat_pred"] = decoded_pred[:, 1].astype(int)
test_df["JobPerspectiveClass_pred"] = decoded_pred[:, 2].astype(int)

# -----------------------------
# JOINT METRICS
# -----------------------------
y_true = test_df["joint"].astype(int)
y_pred = test_df["joint_pred"].astype(int)

acc = accuracy_score(y_true, y_pred)
f1  = f1_score(y_true, y_pred, average="macro")
prec = precision_score(y_true, y_pred, average="macro")
rec  = recall_score(y_true, y_pred, average="macro")
bal = balanced_accuracy_score(y_true, y_pred)

print("\n================= JOINT METRICS =================")
print(f"Accuracy:          {acc:.4f}")
print(f"Macro-F1:          {f1:.4f}")
print(f"Precision (macro): {prec:.4f}")
print(f"Recall (macro):    {rec:.4f}")
print(f"Balanced Accuracy: {bal:.4f}")

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm)
cm_df.to_csv("automl_confusion_matrix.csv", index=False)

# -----------------------------
# PER TARGET METRICS
# -----------------------------
metrics_log = []

def per_target(name, true_col, pred_col):
    yt = test_df[true_col].astype(int)
    yp = test_df[pred_col].astype(int)

    acc = accuracy_score(yt, yp)
    f1  = f1_score(yt, yp, average="macro")
    prec = precision_score(yt, yp, average="macro")
    rec  = recall_score(yt, yp, average="macro")
    bal = balanced_accuracy_score(yt, yp)

    print(f"\n----- {name} -----")
    print(f"Accuracy:      {acc:.4f}")
    print(f"Macro-F1:      {f1:.4f}")
    print(f"Precision:     {prec:.4f}")
    print(f"Recall:        {rec:.4f}")
    print(f"Balanced Acc:  {bal:.4f}")
    report = classification_report(yt, yp, digits=4)
    print(report)

    metrics_log.append(f"\n==== {name} ====\n")
    metrics_log.append(report)

per_target("Frustration", "Frustration", "Frustration_pred")
per_target("JobSat", "JobSat", "JobSat_pred")
per_target("JobPerspectiveClass", "JobPerspectiveClass", "JobPerspectiveClass_pred")

# -----------------------------
# SAVE EVERYTHING
# -----------------------------
# Row-level predictions + features + true targets
test_df.to_csv("automl_joint_predictions.csv", index=False)

# Metrics log as text file
with open("automl_metrics.txt", "w") as f:
    f.write("=== JOINT METRICS ===\n")
    f.write(f"Accuracy: {acc:.4f}\n")
    f.write(f"Macro-F1: {f1:.4f}\n")
    f.write(f"Precision: {prec:.4f}\n")
    f.write(f"Recall: {rec:.4f}\n")
    f.write(f"Balanced Accuracy: {bal:.4f}\n\n")

    for m in metrics_log:
        f.write(m)
        f.write("\n")

print("\n💾 Files generated:")
print("automl_joint_predictions.csv")
print("automl_metrics.txt")
print("automl_confusion_matrix.csv")

# Auto-download (Colab)
files.download("automl_joint_predictions_2025.csv")
files.download("automl_metrics_2025.txt")
files.download("automl_confusion_matrix_2025.csv")


/tmp/ipython-input-747893978.py:37: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  joint_labels, uniques = pd.factorize(tuples)


Number of joint classes: 27
Train size (original): (39298, 17)
Test  size (original): (9825, 17)


No path specified. Models will be saved in: "AutogluonModels/ag-20251202_213548"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Oct  2 10:42:05 UTC 2025
CPU Count:          2
Memory Avail:       9.68 GB / 12.67 GB (76.4%)
Disk Space Avail:   61.32 GB / 107.72 GB (56.9%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `num_stack_le

Train size after SMOTE: (299727, 17)


Leaderboard on holdout data (DyStack):
                    model  score_holdout  score_val        eval_metric  pred_time_test  pred_time_val    fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0  NeuralNetFastAI_BAG_L2       0.743234   0.705196  balanced_accuracy       23.922135      29.490272  827.175559                 1.757365                2.350723         235.735425            2       True          4
1     WeightedEnsemble_L3       0.742903   0.705417  balanced_accuracy       24.331877      30.425108  903.275786                 0.010561                0.165228          18.869525            3       True          6
2  NeuralNetFastAI_BAG_L1       0.683056   0.658853  balanced_accuracy        2.388840       2.001681  416.275182                 2.388840                2.001681         416.275182            1       True          1
3     WeightedEnsemble_L2       0.682125   0.661855  balanced_accuracy       22.178946      2


================= JOINT METRICS =================
Accuracy:          0.4722
Macro-F1:          0.1695
Precision (macro): 0.1725
Recall (macro):    0.1722
Balanced Accuracy: 0.1722

----- Frustration -----
Accuracy:      0.7925
Macro-F1:      0.6572
Precision:     0.6675
Recall:        0.6540
Balanced Acc:  0.6540
              precision    recall  f1-score   support

           0     0.9415    0.9161    0.9286      3671
           1     0.7721    0.8489    0.8087      4925
           2     0.2888    0.1969    0.2342      1229

    accuracy                         0.7925      9825
   macro avg     0.6675    0.6540    0.6572      9825
weighted avg     0.7749    0.7925    0.7816      9825


----- JobSat -----
Accuracy:      0.6910
Macro-F1:      0.5419
Precision:     0.5427
Recall:        0.5451
Balanced Acc:  0.5451
              precision    recall  f1-score   support

           0     0.2339    0.2077    0.2201       491
           1     0.7900    0.7233    0.7552      5964
          

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


FileNotFoundError: Cannot find file: automl_joint_predictions_2025.csv

In [ ]:
# Auto-download (Colab)
files.download("automl_joint_predictions.csv")
files.download("automl_metrics.txt")
files.download("automl_confusion_matrix.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
mapping = pd.DataFrame(
    [(i, tup[0], tup[1], tup[2]) for i, tup in enumerate(uniques)],
    columns=["joint_id", "Frustration", "JobSat", "JobPerspectiveClass"]
)

mapping.to_csv("devx_joint_label_mapping.csv", index=False)
print(mapping)


    joint_id  Frustration  JobSat  JobPerspectiveClass
0          0            1       2                    1
1          1            1       1                    1
2          2            1       1                    0
3          3            2       1                    2
4          4            0       2                    1
5          5            2       0                    1
6          6            0       2                    0
7          7            0       1                    2
8          8            2       1                    1
9          9            0       1                    0
10        10            1       2                    0
11        11            2       2                    0
12        12            2       2                    1
13        13            2       1                    0
14        14            1       2                    2
15        15            1       0                    0
16        16            0       2                    2
17        

2022

In [ ]:
# ===========================================================
# AUTO ML BASELINE — Joint Label Prediction (NO LEAKAGE)
# ===========================================================

!pip install autogluon imbalanced-learn

import os, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, balanced_accuracy_score,
    classification_report, confusion_matrix
)
from imblearn.over_sampling import SMOTE
from google.colab import files
from autogluon.tabular import TabularPredictor

RANDOM_STATE = 42

# -----------------------------
# Load dataset
# -----------------------------
path = "devx_2022_modellingDataset_FLOtop5_union.csv"
df = pd.read_csv(path)

TARGETS = ["MentalHealth", "PurchaseInfluence", "BuyNewTool", ]
FEATURES = [c for c in df.columns if c not in TARGETS]

# basic cleaning
df = df.fillna(df.median(numeric_only=True))
for t in TARGETS:
    df[t] = df[t].astype(int)

# -----------------------------
# Create JOINT label
# -----------------------------
tuples = list(zip(df["MentalHealth"], df["PurchaseInfluence"], df["BuyNewTool"]))
joint_labels, uniques = pd.factorize(tuples)
df["joint"] = joint_labels

print("Number of joint classes:", len(uniques))

# -----------------------------
# Split FIRST (to avoid leakage)
# -----------------------------
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df["joint"]
)

print("Train size (original):", train_df.shape)
print("Test  size (original):", test_df.shape)
# -----------------------------
# Remove ultra-rare JOINT classes (only in train, to fix SMOTE)
# -----------------------------
vc = train_df["joint"].value_counts()
min_required = 4  # needs >= k_neighbors + 1 (here we use k_neighbors=3 later)
rare_labels = vc[vc < min_required].index

print("Rare joint classes being dropped from train:", rare_labels.tolist())
print("Counts of rare classes:\n", vc[vc < min_required])

# Drop those rare joint classes from both train and test
train_df = train_df[~train_df["joint"].isin(rare_labels)].copy()
test_df  = test_df[~test_df["joint"].isin(rare_labels)].copy()

print("Train size after dropping rare classes:", train_df.shape)
print("Test  size after dropping rare classes:", test_df.shape)

# -----------------------------
# SMOTE ONLY ON TRAIN
# -----------------------------
sm = SMOTE(random_state=RANDOM_STATE, k_neighbors=3)

X_train = train_df[FEATURES]
y_train_joint = train_df["joint"]

X_res, y_res = sm.fit_resample(X_train, y_train_joint)

# Decode joint labels back into (M, P, B) just for inspection / metrics
decoded_train = np.array([uniques[i] for i in y_res])

train_bal = pd.DataFrame(X_res, columns=FEATURES)
train_bal["joint"] = y_res
train_bal["MentalHealth"] = decoded_train[:, 0].astype(int)
train_bal["PurchaseInfluence"] = decoded_train[:, 1].astype(int)
train_bal["BuyNewTool"] = decoded_train[:, 2].astype(int)

print("Train size after SMOTE:", train_bal.shape)

# -----------------------------
# Prepare data for AutoML (NO TARGETS AS FEATURES)
# -----------------------------
# AutoGluon sees: FEATURES + joint(label)
train_ag = train_bal.drop(columns=TARGETS)

# For test, we KEEP targets for metrics & CSV, but DROP them for AutoGluon’s input
test_ag = test_df.drop(columns=TARGETS)

# -----------------------------
# AutoML training
# -----------------------------
predictor = TabularPredictor(
    label="joint",
    problem_type="multiclass",
    eval_metric="balanced_accuracy"
).fit(
    train_ag,
    presets="best_quality",   
    time_limit=3600,
    verbosity=2
)

# -----------------------------
# Predictions on TEST (no leakage)
# -----------------------------
test_df = test_df.copy()  # to be explicit

# Predict using the leakage-free test_ag (no target columns)
test_df["joint_pred"] = predictor.predict(test_ag)

# Decode joint predictions -> per-target predictions
joint_pred_int = test_df["joint_pred"].astype(int).to_numpy()
decoded_pred = np.array([uniques[i] for i in joint_pred_int])

test_df["MentalHealth_pred"] = decoded_pred[:, 0].astype(int)
test_df["PurchaseInfluence_pred"] = decoded_pred[:, 1].astype(int)
test_df["BuyNewTool_pred"] = decoded_pred[:, 2].astype(int)

# -----------------------------
# JOINT METRICS
# -----------------------------
y_true = test_df["joint"].astype(int)
y_pred = test_df["joint_pred"].astype(int)

acc = accuracy_score(y_true, y_pred)
f1  = f1_score(y_true, y_pred, average="macro")
prec = precision_score(y_true, y_pred, average="macro")
rec  = recall_score(y_true, y_pred, average="macro")
bal = balanced_accuracy_score(y_true, y_pred)

print("\n================= JOINT METRICS =================")
print(f"Accuracy:          {acc:.4f}")
print(f"Macro-F1:          {f1:.4f}")
print(f"Precision (macro): {prec:.4f}")
print(f"Recall (macro):    {rec:.4f}")
print(f"Balanced Accuracy: {bal:.4f}")

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm)
cm_df.to_csv("automl_confusion_matrix.csv", index=False)

# -----------------------------
# PER TARGET METRICS
# -----------------------------
metrics_log = []

def per_target(name, true_col, pred_col):
    yt = test_df[true_col].astype(int)
    yp = test_df[pred_col].astype(int)

    acc = accuracy_score(yt, yp)
    f1  = f1_score(yt, yp, average="macro")
    prec = precision_score(yt, yp, average="macro")
    rec  = recall_score(yt, yp, average="macro")
    bal = balanced_accuracy_score(yt, yp)

    print(f"\n----- {name} -----")
    print(f"Accuracy:      {acc:.4f}")
    print(f"Macro-F1:      {f1:.4f}")
    print(f"Precision:     {prec:.4f}")
    print(f"Recall:        {rec:.4f}")
    print(f"Balanced Acc:  {bal:.4f}")
    report = classification_report(yt, yp, digits=4)
    print(report)

    metrics_log.append(f"\n==== {name} ====\n")
    metrics_log.append(report)

per_target("MentalHealth", "MentalHealth", "MentalHealth_pred")
per_target("PurchaseInfluence", "PurchaseInfluence", "PurchaseInfluence_pred")
per_target("BuyNewTool", "BuyNewTool", "BuyNewTool_pred")

# -----------------------------
# SAVE EVERYTHING
# -----------------------------
# Row-level predictions + features + true targets
test_df.to_csv("automl_joint_predictions.csv", index=False)

# Metrics log as text file
with open("automl_metrics.txt", "w") as f:
    f.write("=== JOINT METRICS ===\n")
    f.write(f"Accuracy: {acc:.4f}\n")
    f.write(f"Macro-F1: {f1:.4f}\n")
    f.write(f"Precision: {prec:.4f}\n")
    f.write(f"Recall: {rec:.4f}\n")
    f.write(f"Balanced Accuracy: {bal:.4f}\n\n")

    for m in metrics_log:
        f.write(m)
        f.write("\n")

print("\n💾 Files generated:")
print("automl_joint_predictions.csv")
print("automl_metrics.txt")
print("automl_confusion_matrix.csv")

# Auto-download (Colab)
files.download("automl_joint_predictions.csv")
files.download("automl_metrics.txt")
files.download("automl_confusion_matrix.csv")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 9.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking

/tmp/ipython-input-1618110694.py:37: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  joint_labels, uniques = pd.factorize(tuples)


Number of joint classes: 27
Train size (original): (58614, 17)
Test  size (original): (14654, 17)
Rare joint classes being dropped from train: [26]
Counts of rare classes:
 joint
26    2
Name: count, dtype: int64
Train size after dropping rare classes: (58612, 17)
Test  size after dropping rare classes: (14653, 17)


No path specified. Models will be saved in: "AutogluonModels/ag-20251208_121032"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Oct  2 10:42:05 UTC 2025
CPU Count:          2
Memory Avail:       11.02 GB / 12.67 GB (86.9%)
Disk Space Avail:   63.15 GB / 107.72 GB (58.6%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `num_stack_l

Train size after SMOTE: (301314, 17)


	Running DyStack sub-fit in a ray process to avoid memory leakage. Enabling ray logging (enable_ray_logging=True). Specify `ds_args={'enable_ray_logging': False}` if you experience logging issues.
2025-12-08 12:10:35,850	INFO worker.py:1843 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
		Context path: "/content/AutogluonModels/ag-20251208_121032/ds_sub_fit/sub_fit_ho"
(_dystack pid=3640) Running DyStack sub-fit ...
(_dystack pid=3640) Beginning AutoGluon training ... Time limit = 894s
(_dystack pid=3640) AutoGluon will save models to "/content/AutogluonModels/ag-20251208_121032/ds_sub_fit/sub_fit_ho"
(_dystack pid=3640) Train Data Rows:    267834
(_dystack pid=3640) Train Data Columns: 13
(_dystack pid=3640) Label Column:       joint
(_dystack pid=3640) Problem Type:       multiclass
(_dystack pid=3640) Preprocessing data ...
(_dystack pid=3640) Train Data Class Count: 26
(_dystack pid=3640) Using Feature Generators to preprocess the data ...
(_dystack p


================= JOINT METRICS =================
Accuracy:          0.2657
Macro-F1:          0.1706
Precision (macro): 0.1681
Recall (macro):    0.1976
Balanced Accuracy: 0.1976

----- MentalHealth -----
Accuracy:      0.6829
Macro-F1:      0.4351
Precision:     0.4210
Recall:        0.4680
Balanced Acc:  0.4680
              precision    recall  f1-score   support

           0     0.8280    0.7934    0.8103     11540
           1     0.2559    0.2593    0.2576      2649
           2     0.1791    0.3513    0.2373       464

    accuracy                         0.6829     14653
   macro avg     0.4210    0.4680    0.4351     14653
weighted avg     0.7040    0.6829    0.6923     14653


----- PurchaseInfluence -----
Accuracy:      0.6633
Macro-F1:      0.6536
Precision:     0.6414
Recall:        0.6757
Balanced Acc:  0.6757
              precision    recall  f1-score   support

           0     0.6149    0.7141    0.6608      4428
           1     0.7458    0.6278    0.6817      781

FileNotFoundError: Cannot find file: automl_joint_predictions_2025.csv